# Emotion in CMV opening replies

Can a pretrained classifier identify emotions expressed by CMV reply authors well enough to justify testing them in P7? This pilot uses only unique opening replies from the matched training pairs.

The checkpoint is [SamLowe/roberta-base-go_emotions](https://huggingface.co/SamLowe/roberta-base-go_emotions/tree/9045a3949b4d643baee6296673cea5bbfdc66c04). It supplies 28 independent emotion scores. The manual review focuses on joy, anger, sadness, fear, surprise, disgust, and neutral.

We use the cleaned openings from the paired linguistic analysis. Each long opening is split into chunks, and each emotion's reply-level score is the maximum over its chunks. A score of at least 0.5 counts as a detection in this pilot. Longer replies have more chances to contain a high-scoring chunk. Classifier scores are proxies for detected language, with no calibration or intensity interpretation assumed. Neutral is a model label, and an opening can also receive no labels above threshold.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "project" / "notebooks" / "linguistic-features" / "emotion").is_dir()
)
pilot_root = root / "project/data/processed/linguistic-features/emotion-validation-pilot"
run_id = None  # Set to a specific run folder name to reopen it later.
runs = sorted(path.parent for path in pilot_root.glob("*/pilot_manifest.json"))

if run_id is None and not runs:
    raise FileNotFoundError("Run score_emotion_validation_pilot.py to generate the pilot files.")

run = pilot_root / run_id if run_id else runs[-1]
manifest = json.loads((run / "pilot_manifest.json").read_text())
review_path = run / "blind_emotion_review.csv"
labels = manifest["review_labels"]

print(f"Run {run.name}")
print(f'{manifest["candidate_count"]} scored training openings · {manifest["review_count"]} selected for review')
print(f"Blind review CSV · {review_path}")

## Review the sample

Open `blind_emotion_review.csv` and fill in `manual_labels`. Use one or more of `joy`, `anger`, `sadness`, `fear`, `surprise`, or `disgust`, separated by a pipe such as `anger|disgust`. Use `other` for emotions outside the list, `neutral` when the author expresses no discernible emotion, and `unclear` if you cannot judge. Use `neutral` and `unclear` on their own. Leave unfinished rows blank.

Label emotions expressed by the author in the displayed text. Use `review_notes` to flag attributed emotion, inline quotations, sarcasm, or negation. The shared text cleaner already removes Markdown block quotes, so the review covers what the model actually saw. Model predictions and delta outcomes are omitted from the review sheet.

The sample contains a random subset plus replies enriched for candidate labels, including high-scoring below-threshold examples when detections were scarce. Agreement here describes the reviewed examples. It should not be generalized to the full corpus.

In [ ]:
review = pd.read_csv(review_path, keep_default_na=False)
scores = pd.read_parquet(run / "candidate_emotion_scores.parquet").rename_axis("comment_id")

if not review["comment_id"].is_unique:
    raise ValueError("The review CSV must have one row per comment ID.")

review["labels"] = review["manual_labels"].map(
    lambda value: {label.strip().lower() for label in value.split("|") if label.strip()}
)
allowed = set(labels) | {"other", "unclear"}
invalid = review["labels"].map(
    lambda values: bool(values - allowed) or (len(values) > 1 and bool(values & {"neutral", "unclear"}))
)
if invalid.any():
    raise ValueError(f"Check the labels for {', '.join(review.loc[invalid, 'review_id'])}.")

labeled = review["labels"].map(lambda values: bool(values) and values != {"unclear"})
print(f"{labeled.sum()} of {len(review)} replies have usable manual labels.")

if labeled.any():
    reviewed = review.loc[labeled].join(scores, on="comment_id", validate="one_to_one")
    if reviewed["chunk_count"].isna().any():
        raise ValueError("Manual review comment IDs must match the scoring run.")
    rows = []
    for label in labels:
        human = reviewed["labels"].map(lambda values: label in values)
        model = reviewed[f"detected_{label}"]
        rows.append({
            "emotion": label,
            "human positives": int(human.sum()),
            "model positives": int(model.sum()),
            "both positive": int((human & model).sum()),
            "model only": int((~human & model).sum()),
            "human only": int((human & ~model).sum()),
        })
    display(pd.DataFrame(rows).set_index("emotion"))

## Decision after review

If the labels look usable on CMV, score the full training cohort and compare prevalence by outcome and discordant-pair win share. Reuse the 50-word sensitivity subset and discussion-clustered intervals. Then compare emotion-only E0, P6, and P7 on the frozen discussion-grouped folds, using continuous A-minus-B emotion scores and fitting preprocessing inside each training fold.

Prioritize the changes in ROC-AUC and log loss, with average precision and accuracy alongside them. A gain of roughly 0.01 ROC-AUC with an interval above zero is the proposed practical benchmark for advancing P7. A log-loss gain can still justify examining probability quality. Revisit held-out evaluation and error-transition visualizations after reviewing those results.